<a href="https://colab.research.google.com/github/Ajinkya718/portfo/blob/main/extras/exercises/05_pytorch_going_modular_exercise_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05. PyTorch Going Modular Exercises

Welcome to the 05. PyTorch Going Modular exercise template notebook.

There are several questions in this notebook and it's your goal to answer them by writing Python and PyTorch code.

> **Note:** There may be more than one solution to each of the exercises, don't worry too much about the *exact* right answer. Try to write some code that works first and then improve it if you can.

## Resources and solutions

* These exercises/solutions are based on [section 05. PyTorch Going Modular](https://www.learnpytorch.io/05_pytorch_going_modular/) of the Learn PyTorch for Deep Learning course by Zero to Mastery.

**Solutions:**

Try to complete the code below *before* looking at these.

* See a live [walkthrough of the solutions (errors and all) on YouTube](https://youtu.be/ijgFhMK3pp4).
* See an example [solutions notebook for these exercises on GitHub](https://github.com/mrdbourke/pytorch-deep-learning/blob/main/extras/solutions/05_pytorch_going_modular_exercise_solutions.ipynb).

## 1. Turn the code to get the data (from section 1. Get Data) into a Python script, such as `get_data.py`.

* When you run the script using `python get_data.py` it should check if the data already exists and skip downloading if it does.
* If the data download is successful, you should be able to access the `pizza_steak_sushi` images from the `data` directory.

In [1]:
# YOUR CODE HERE
%%writefile get_data.py
import os
import zipfile
import requests
from pathlib import Path

data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)

with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
    request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
    print("Downloading pizza, steak, sushi data...")
    f.write(request.content)

with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
    print("Unzipping pizza, steak, sushi data...")
    zip_ref.extractall(image_path)

os.remove(data_path / "pizza_steak_sushi.zip")

Writing get_data.py


In [2]:
# Example running of get_data.py
!python get_data.py

Did not find data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


## 2. Use [Python's `argparse` module](https://docs.python.org/3/library/argparse.html) to be able to send the `train.py` custom hyperparameter values for training procedures.
* Add an argument flag for using a different:
  * Training/testing directory
  * Learning rate
  * Batch size
  * Number of epochs to train for
  * Number of hidden units in the TinyVGG model
    * Keep the default values for each of the above arguments as what they already are (as in notebook 05).
* For example, you should be able to run something similar to the following line to train a TinyVGG model with a learning rate of 0.003 and a batch size of 64 for 20 epochs: `python train.py --learning_rate 0.003 batch_size 64 num_epochs 20`.
* **Note:** Since `train.py` leverages the other scripts we created in section 05, such as, `model_builder.py`, `utils.py` and `engine.py`, you'll have to make sure they're available to use too. You can find these in the [`going_modular` folder on the course GitHub](https://github.com/mrdbourke/pytorch-deep-learning/tree/main/going_modular/going_modular).

In [6]:
# YOUR CODE HERE
%%writefile /content/train.py
import subprocess
import sys
from pathlib import Path

repo_path = Path("/content/pytorch-deep-learning")

if not repo_path.exists():
    subprocess.run(["git", "clone", "https://github.com/mrdbourke/pytorch-deep-learning.git"])

sys.path.append(str(repo_path / "going_modular" / "going_modular"))

import torch
import data_setup, engine, model_builder, utils
import argparse
from torchvision import transforms

parser = argparse.ArgumentParser(description="Train TinyVGG model")

parser.add_argument("--train_dir", type = str, default = "data/pizza_steak_sushi/train")
parser.add_argument("--test_dir", type = str, default="data/pizza_steak_sushi/test")
parser.add_argument("--num_epochs", type = int, default = 5)
parser.add_argument("--batch_size", type = int, default = 32)
parser.add_argument("--hidden_units", type = int, default = 10)
parser.add_argument("--learning_rate", type = float, default = 0.001)

args, unknown = parser.parse_known_args()

device = "cuda" if torch.cuda.is_available() else "cpu"

data_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir = args.train_dir,
    test_dir = args.test_dir,
    transform = data_transform,
    batch_size = args.batch_size
)

model = model_builder.TinyVGG(
    input_shape = 3,
    hidden_units = args.hidden_units,
    output_shape = len(class_names)
).to(device)

loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = args.learning_rate)

engine.train(
    model = model,
    train_dataloader = train_dataloader,
    test_dataloader = test_dataloader,
    loss_fn = loss_fn,
    optimizer = optimizer,
    epochs = args.num_epochs,
    device = device
)

utils.save_model(
    model = model,
    target_dir = "models",
    model_name = "05_going_modular_script_mode_tinyvgg_model.pth"
)

Writing /content/train.py


In [7]:
# Example running of train.py
!python /content/train.py --num_epochs 5 --batch_size 128 --hidden_units 128 --learning_rate 0.0003

  0% 0/5 [00:00<?, ?it/s]Epoch: 1 | train_loss: 1.0983 | train_acc: 0.3315 | test_loss: 1.0925 | test_acc: 0.3333
 20% 1/5 [00:01<00:05,  1.43s/it]Epoch: 2 | train_loss: 1.0860 | train_acc: 0.3384 | test_loss: 1.0712 | test_acc: 0.3733
 40% 2/5 [00:02<00:04,  1.41s/it]Epoch: 3 | train_loss: 1.0576 | train_acc: 0.4937 | test_loss: 1.0499 | test_acc: 0.5067
 60% 3/5 [00:04<00:02,  1.47s/it]Epoch: 4 | train_loss: 1.0155 | train_acc: 0.5166 | test_loss: 1.0192 | test_acc: 0.4800
 80% 4/5 [00:05<00:01,  1.30s/it]Epoch: 5 | train_loss: 0.9389 | train_acc: 0.5866 | test_loss: 1.0416 | test_acc: 0.4133
100% 5/5 [00:06<00:00,  1.30s/it]
[INFO] Saving model to: models/05_going_modular_script_mode_tinyvgg_model.pth


## 3. Create a Python script to predict (such as `predict.py`) on a target image given a file path with a saved model.

* For example, you should be able to run the command `python predict.py some_image.jpeg` and have a trained PyTorch model predict on the image and return its prediction.
* To see example prediction code, check out the [predicting on a custom image section in notebook 04](https://www.learnpytorch.io/04_pytorch_custom_datasets/#113-putting-custom-image-prediction-together-building-a-function).
* You may also have to write code to load in a trained model.

In [14]:
# YOUR CODE HERE
%%writefile /content/predict.py

import subprocess
import sys
from pathlib import Path

repo_path = Path("/content/pytorch-deep-learning")

if not repo_path.exists():
    subprocess.run(["git", "clone", "https://github.com/mrdbourke/pytorch-deep-learning.git"],
                   check=True)

sys.path.append(str(repo_path / "going_modular" / "going_modular"))

import argparse
import torch
import model_builder
from torchvision import transforms
from PIL import Image

parser = argparse.ArgumentParser(description="Predicting on a target image")
parser.add_argument("--image", type = str, help = "Path to the target image")
parser.add_argument("--model_path", type = str, default="models/05_going_modular_script_mode_tinyvgg_model.pth", help = "Path to the trained model")
parser.add_argument("--class_names", type = str, nargs="+", default=["pizza", "steak", "sushi"], help = "List of class names")
parser.add_argument("--hidden_units", type = int, default = 128)
parser.add_argument("--image_size", type = int, default = 64)

args, unknown = parser.parse_known_args()

device = "cuda" if torch.cuda.is_available() else "cpu"

model = model_builder.TinyVGG(
    input_shape = 3,
    hidden_units = args.hidden_units,
    output_shape = len(args.class_names)).to(device)

model.load_state_dict(torch.load(args.model_path, map_location=device))

transform = transforms.Compose([
    transforms.Resize((args.image_size, args.image_size)),
    transforms.ToTensor()
])

image = Image.open(args.image)
transformed_image = transform(image).unsqueeze(0).to(device)

model.eval()
with torch.inference_mode():
    target_image_pred = model(transformed_image)
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)

print(f"Predcition class: {args.class_names[target_image_pred_label]}")
print(f"Prediction probability: {target_image_pred_probs.max():.3f}")

Overwriting /content/predict.py


In [15]:
# Example running of predict.py
!python /content/predict.py --image data/pizza_steak_sushi/test/sushi/175783.jpg

Predcition class: pizza
Prediction probability: 0.513
